# Fetal ECG Extraction — 1D U-Net Pipeline

Extracts fetal ECG from 4-channel abdominal recordings using a 1D U-Net
trained with a hybrid MSE + correlation loss.

## 1. Visualize Raw Signals

In [ ]:
from utils import visualize_signals

visualize_signals('data/r01.edf', start_sec=10, duration_sec=4)
visualize_signals('data/r10.edf', start_sec=10, duration_sec=4)

## 2. Load and Preprocess Data

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from utils import get_dataloaders

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

train_files = ['data/r01.edf', 'data/r04.edf', 'data/r07.edf', 'data/r08.edf']
test_files = ['data/r10.edf']

full_train_loader, test_loader = get_dataloaders(train_files, test_files, batch_size=8)
full_dataset = full_train_loader.dataset

total_count = len(full_dataset)
val_count = int(total_count * 0.2)
train_count = total_count - val_count

train_dataset, val_dataset = random_split(full_dataset, [train_count, val_count])
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

print(f"Training:   {len(train_loader)} batches")
print(f"Validation: {len(val_loader)} batches")
print(f"Test:       {len(test_loader)} batches")

## 3. Model Definition — 1D U-Net

In [ ]:
import torch.nn.functional as F


class UNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv1d(4, 16, 7, padding=3), nn.BatchNorm1d(16), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv1d(16, 32, 5, padding=2), nn.BatchNorm1d(32), nn.ReLU())
        self.enc3 = nn.Sequential(nn.Conv1d(32, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU())
        self.dec1 = nn.Sequential(nn.Conv1d(96, 32, 3, padding=1), nn.BatchNorm1d(32), nn.ReLU())
        self.dec2 = nn.Sequential(nn.Conv1d(48, 16, 5, padding=2), nn.BatchNorm1d(16), nn.ReLU())
        self.final = nn.Conv1d(16, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        bottleneck = self.enc3(e2)
        d1 = self.dec1(torch.cat([bottleneck, e2], dim=1))
        d2 = self.dec2(torch.cat([d1, e1], dim=1))
        return self.final(d2)


class HybridLoss(nn.Module):
    """MSE + (1 - Pearson correlation) loss."""
    def __init__(self, alpha=0.5):
        super().__init__()
        self.mse = nn.MSELoss()
        self.alpha = alpha

    def forward(self, pred, target):
        mse_loss = self.mse(pred, target)
        pred_flat = pred.view(pred.shape[0], -1)
        target_flat = target.view(target.shape[0], -1)
        pred_mean = pred_flat - torch.mean(pred_flat, dim=1, keepdim=True)
        target_mean = target_flat - torch.mean(target_flat, dim=1, keepdim=True)
        numerator = torch.sum(pred_mean * target_mean, dim=1)
        denominator = (
            torch.sqrt(torch.sum(pred_mean**2, dim=1))
            * torch.sqrt(torch.sum(target_mean**2, dim=1))
        )
        corr = numerator / (denominator + 1e-8)
        corr_loss = 1 - torch.mean(corr)
        return (1 - self.alpha) * mse_loss + self.alpha * corr_loss


model = UNet1D().to(device)
print(f"UNet1D parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Training

In [ ]:
import time

criterion = HybridLoss(alpha=0.5)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=3
)

num_epochs = 20
print(f"Training on {device} for {num_epochs} epochs")
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    scheduler.step(avg_loss)
    lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {avg_loss:.6f} | LR: {lr:.6f}")

print(f"Training complete in {time.time() - start_time:.1f}s")

## 5. Waveform Comparison (Predicted vs. Ground Truth)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

abdominal_data, scalp_data = next(iter(test_loader))
abdominal_data = abdominal_data.to(device)
model.eval()
with torch.no_grad():
    prediction = model(abdominal_data)

idx = 0
pred_wave = prediction[idx, 0, :].cpu().numpy()
true_wave = scalp_data[idx, 0, :].cpu().numpy()

fig, axs = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axs[0].plot(true_wave, color='green', label='Actual Fetal Scalp')
axs[0].set_title(f'Ground Truth | Window {idx}')
axs[0].legend(loc='upper right')
axs[0].grid(True, alpha=0.3)

axs[1].plot(pred_wave, color='blue', label='Predicted Output')
axs[1].set_title('Model Prediction')
axs[1].legend(loc='upper right')
axs[1].grid(True, alpha=0.3)

axs[2].plot(true_wave, color='green', alpha=0.5, label='Actual', linestyle='--')
axs[2].plot(pred_wave, color='blue', alpha=0.8, label='Predicted')
axs[2].set_title('Overlay')
axs[2].set_xlabel('Time Steps')
axs[2].legend(loc='upper right')
axs[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Test Set Evaluation

In [ ]:
model.eval()
total_mse = total_mae = total_corr = 0.0
count = 0

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)

        total_mse += torch.mean((outputs - targets) ** 2).item()
        total_mae += torch.mean(torch.abs(outputs - targets)).item()

        pred_flat = outputs.cpu().numpy().reshape(outputs.shape[0], -1)
        true_flat = targets.cpu().numpy().reshape(targets.shape[0], -1)
        batch_corr = 0
        for i in range(len(pred_flat)):
            c = np.corrcoef(pred_flat[i], true_flat[i])[0, 1]
            if not np.isnan(c):
                batch_corr += c
        total_corr += batch_corr / len(pred_flat)
        count += 1

print(f"Test results over {len(test_loader.dataset)} windows:")
print(f"  MSE:         {total_mse/count:.6f}")
print(f"  MAE:         {total_mae/count:.6f}")
print(f"  Pearson r:   {total_corr/count:.4f}")

## 7. Clinical Validation — Heart Rate (BPM)

In [ ]:
from scipy.signal import find_peaks

model.eval()
inputs, targets = next(iter(test_loader))
inputs = inputs.to(device)

with torch.no_grad():
    outputs = model(inputs)

for i in range(min(5, len(inputs))):
    pred_wave = outputs[i, 0, :].cpu().numpy()
    true_wave = targets[i, 0, :].cpu().numpy()

    pred_peaks, _ = find_peaks(pred_wave, distance=300, height=np.max(pred_wave) * 0.4)
    true_peaks, _ = find_peaks(true_wave, distance=300, height=np.max(true_wave) * 0.4)

    pred_bpm = len(pred_peaks) * 30
    true_bpm = len(true_peaks) * 30
    bpm_error = abs(pred_bpm - true_bpm)
    status = 'PASS' if bpm_error <= 15 else 'CHECK'

    print(f"Window {i}: True {true_bpm} BPM | Pred {pred_bpm} BPM | Diff {bpm_error} -> {status}")

## 8. Frequency Domain Comparison (PSD)

In [ ]:
from scipy.signal import welch

idx = 0
pred_wave = outputs[idx, 0, :].cpu().numpy()
true_wave = targets[idx, 0, :].cpu().numpy()

fs = 1000
f_true, Pxx_true = welch(true_wave, fs=fs, nperseg=1024)
f_pred, Pxx_pred = welch(pred_wave, fs=fs, nperseg=1024)

plt.figure(figsize=(10, 6))
plt.semilogy(f_true, Pxx_true, color='green', label='Ground Truth PSD')
plt.semilogy(f_pred, Pxx_pred, color='blue', label='Predicted PSD')
plt.title('Frequency Domain Comparison (Power Spectral Density)')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power/Frequency (dB/Hz)')
plt.xlim(0, 100)
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.show()